# LEC: task-phase 4-periodicity in the population

Binary state decoders don't generalise across tasks — state **identity** remaps
(`CCGP_STATE_PAIRS.md`). So instead: is there **4-fold periodic structure** in the
population around the A→B→C→D→A loop — a signal that recurs at the rhythm of the
four states — **without** assuming it aligns to the tone at A (phase-free)?

"4-fold" = as task phase runs once round the loop, a component completing **4
cycles** (one per state) = power at **harmonic 4** of the 360-bin trajectory.
Contrast h=1 (the 4 states on one ring) and h=2 (A/C vs B/D alternation).

Pilot (read-only) already showed h=4 dominates but ~half of it sits at the reward
boundaries and the four legs barely repeat — so the question is **what drives it**.
Method + rationale: `TASKPHASE_PERIODICITY.md`. All logic is in
`taskphase_periodicity.py` (identical copy under `mFC_data/code/`).

In [ ]:
import sys
sys.path.insert(0, ".")
from importlib import reload
import numpy as np, pandas as pd, pickle, os
import matplotlib.pyplot as plt
import glm_analysis_v2, taskphase_periodicity as tp
reload(glm_analysis_v2); reload(tp)
from glm_analysis_v2 import apply_gridmaze_style
apply_gridmaze_style()

REGION = "LEC"
SAVE_DIR = "../data/figures/taskphase_periodicity"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# ── LEC data ────────────────────────────────────────────────────────────────
Data_folder = "../data/processed_data"
neuron_folder = f"{Data_folder}/neuron_raw_mingyutest"
with open(os.path.join(Data_folder, "data_dic_lec.pkl"), "rb") as f:
    data_dic = pickle.load(f)
neuron_files = [f for f in os.listdir(neuron_folder)
                if f.startswith("Neuron_raw_") and f.endswith(".npy")]
recdays = sorted({"_".join(f[len("Neuron_raw_"):-len(".npy")].split("_")[:-1])
                  for f in neuron_files})
mouse_recdays = [mr for mr in recdays if mr in data_dic and "_sb" not in mr]
print(len(mouse_recdays), "recdays,", len({m[:4] for m in mouse_recdays}), "mice")

## 1. Synthetic controls — the gate

Five synthetic populations flow through the **real** pipeline. The gate is that the
metrics **discriminate** them — if the pipeline can't tell a genuine cycle from a
ramp / a reward transient / warp-floor state selectivity, it can't interpret the
real data.

| synthetic | expected |
|---|---|
| `cycle` (pure h4) | high h4/base, decay≈0, cross-task progress generalises |
| `ramp` (progress) | decay≈0.25 (arc), survives trim |
| `boundary` (reward transient) | h4/base collapses under trim; phase at reward |
| `state` (state-selective) | h4 **not** elevated, leg-corr≈0 — **the warp floor** |
| `noise` | flat |

In [ ]:
reload(tp)
synth = tp.run_synthetic_controls()
synth.round(3)

## 2. Headline — the phase-free harmonic spectrum

Per-neuron `rfft` power around the 360-bin loop, averaged over neurons and tasks.
The existence test is the within-spectrum comparison: is h=4 (and its multiples)
elevated over the non-multiple-of-4 harmonics? Phase coherence (do neurons share a
common h4 phase) uses the circular-roll null — which **only** answers the phase
question, since it preserves each neuron's power spectrum.

In [ ]:
config = tp.PeriodicityConfig()
print(config)
res = tp.run_spectrum_batch(data_dic, config, mouse_recdays=mouse_recdays)
spectrum, summary = res["spectrum"], res["summary"]
spectrum.to_pickle(f"{SAVE_DIR}/spectrum_{REGION}.pkl")
summary.to_pickle(f"{SAVE_DIR}/summary_{REGION}.pkl")
print(f"\nh4/baseline ratio: mean {summary['h_target_ratio'].mean():.2f} "
      f"(median {summary['h_target_ratio'].median():.2f}); "
      f"h4 top harmonic in {summary['frac_target_top'].mean()*100:.0f}% of recdays")
print(f"phase coherence R {summary['phase_coherence_R'].mean():.3f} "
      f"vs null {summary['phase_coherence_null'].mean():.3f}")
tp.plot_spectrum(spectrum, region=REGION, config=config,
                 out_path=f"{SAVE_DIR}/spectrum_{REGION}.pdf"); plt.show()

## 3. Confound control C1 — reward-boundary trimming

How much of h4 is a reward transient vs a genuine mid-leg cycle? Trim the reward
windows off each leg end. A boundary transient dies; a mid-leg cycle survives.

In [ ]:
trim = pd.concat([tp.spectrum_vs_trim(data_dic, mr, config, trims=(0, 10, 20, 30))
                  for mr in mouse_recdays], ignore_index=True)
trim.to_pickle(f"{SAVE_DIR}/trim_{REGION}.pkl")
display(trim.groupby("trim_pct")["ratio"].agg(["mean", "sem"]).round(2))
tp.plot_trim(trim, region=REGION, out_path=f"{SAVE_DIR}/trim_{REGION}.pdf"); plt.show()

## 4. Confound controls C2 (raw-time) & C3 (ramp vs cycle) + phase / state-invariance

C2: an independent spectrum in **raw time** (no warping) via a cos/sin GLM on the
continuous task phase — confirms h4 isn't a warping artifact. C3: is the sub-goal
structure a smooth **cycle** or a goal-progress **ramp** (decay ratio + openness)?
Plus per-neuron h4 phase (mid-leg vs reward) and leg-to-leg similarity.

In [ ]:
rows = []
for mr in mouse_recdays:
    tasks = tp.build_taskphase_curves(data_dic, mr, config)
    if not tasks:
        continue
    raw = [tp.raw_time_spectrum(t["session_data"], config)[0] for t in tasks]
    geo = tp.ramp_vs_cycle(tasks, config)
    ph = tp.phase_analysis(tasks, config)
    rows.append({"mouse_recday": mr, "mouse": mr[:4],
                 "raw_time_ratio": np.nanmean(raw), **geo, **ph})
chars = pd.DataFrame(rows)
chars.to_pickle(f"{SAVE_DIR}/chars_{REGION}.pkl")
chars[["raw_time_ratio", "decay_ratio", "openness", "leg_corr",
       "phase_R", "phase_mean_frac", "frac_amp_at_reward",
       "per_leg_phase_consistency"]].mean().round(3)

## 5. Per-neuron periodic cells

How many single cells have h4 as an outlier in their own spectrum, and where do
they peak within the leg (grid-cell-style phase rose).

In [ ]:
cells = []
for mr in mouse_recdays:
    tasks = tp.build_taskphase_curves(data_dic, mr, config)
    if not tasks:
        continue
    c = tp.periodic_cells(tasks, config)
    c["mouse_recday"] = mr; c["mouse"] = mr[:4]
    cells.append(c)
cells = pd.concat(cells, ignore_index=True)
cells.to_pickle(f"{SAVE_DIR}/cells_{REGION}.pkl")
print(f"{cells['is_periodic'].mean()*100:.0f}% of {len(cells)} cells are 4-periodic")
tp.plot_phase_rose(cells, region=REGION, out_path=f"{SAVE_DIR}/phase_rose_{REGION}.pdf"); plt.show()

## 6. Cross-task generalisation — the CCGP sequel

State identity remaps. Does within-leg **progress** generalise across tasks?
Leave-one-task-out progress decoding, pooled over the four states, vs a
role-permutation null. progress > null while state-CCGP ≈ null ⇒ *which-goal
remaps, but progress-through-goal is abstract*. (~10–20 min: LinearSVC + null.)

In [ ]:
prog = pd.concat([tp.run_cross_task_progress(data_dic, mr, config)
                  for mr in mouse_recdays], ignore_index=True)
prog.to_pickle(f"{SAVE_DIR}/progress_{REGION}.pkl")
rec = prog.groupby(["mouse_recday", "mouse"]).agg(
    acc=("progress_acc", "mean"), null=("null_mean", "mean"),
    chance=("chance", "first")).reset_index()
from scipy import stats
print(f"progress acc {rec['acc'].mean():.3f} vs null {rec['null'].mean():.3f} "
      f"(chance {rec['chance'].iloc[0]:.3f}); "
      f"Wilcoxon p={stats.wilcoxon(rec['acc'], rec['null']).pvalue:.2g}")
tp.plot_progress_ccgp(prog, region=REGION, out_path=f"{SAVE_DIR}/progress_{REGION}.pdf"); plt.show()

## 8. Ring structure — is there anything ring-like? (whole-loop h1)

A whole-trial ring = the **1st harmonic** (one cycle per ABCD loop). `ring_analysis`
reports h1 vs its neighbours (h2,h3) and the trajectory **winding number** (1 = one
ring, 4 = four-fold). Expected from the spectrum + PH: no ring.

In [ ]:
ring = pd.DataFrame([dict(mouse_recday=mr, mouse=mr[:4],
                         **tp.ring_analysis(tp.build_taskphase_curves(data_dic, mr, config), config))
                     for mr in mouse_recdays
                     if tp.build_taskphase_curves(data_dic, mr, config)])
ring.to_pickle(f"{SAVE_DIR}/ring_{REGION}.pkl")
print(f"h1/neighbour {ring['h1_ratio'].mean():.2f} (a ring needs >>1), "
      f"h4/neighbour {ring['h4_ratio'].mean():.2f}, winding {ring['winding'].mean():.2f}")
ring[['h1_ratio','h4_ratio','winding']].mean().round(2)

## 9. How much of the cross-task progress is spatial?

**#1 place-matched split** (pairwise progress vs reward-tower overlap) and
**#2/#3 cross-task variable decoding** (time-progress / distance-progress / location /
state, on the raw-time `prepare_session_data` outputs). Location is the negative
control — but note the maze corridors are shared across tasks, so location partly
generalises too. (Slow: `prepare_session_data` per session.)

In [ ]:
split = pd.concat([tp.run_progress_place_split(data_dic, mr, config)
                   for mr in mouse_recdays], ignore_index=True)
split.to_pickle(f"{SAVE_DIR}/placesplit_{REGION}.pkl")
lo = split[split['tower_overlap']==0]['progress_acc']
hi = split[split['tower_overlap']>0]['progress_acc']
print(f"progress acc: 0 tower overlap {lo.mean():.3f} (n={len(lo)}) vs shared {hi.mean():.3f} (n={len(hi)})")
tp.plot_place_split(split, region=REGION, out_path=f"{SAVE_DIR}/placesplit_{REGION}.pdf"); plt.show()

In [ ]:
var = pd.concat([tp.run_cross_task_variables(data_dic, mr, config)
                 for mr in mouse_recdays], ignore_index=True)
var.to_pickle(f"{SAVE_DIR}/variables_{REGION}.pkl")
r = var.groupby(['mouse_recday','variable']).agg(acc=('acc','mean'), null=('null_mean','mean')).reset_index()
r['above'] = r['acc'] - r['null']
display(r.groupby('variable')[['acc','null','above']].mean().round(3))
tp.plot_cross_task_variables(var, region=REGION, out_path=f"{SAVE_DIR}/variables_{REGION}.pdf"); plt.show()

## 10. Grid-cell generalisation — coherent re-anchoring across tasks

A grid code re-anchors coherently (global phase shift, preserved cell-to-cell
offsets), unlike place/state which remap independently. `grid_generalization` tests
the per-cell h4 phase coherence across task pairs vs a cell-shuffle null.

In [ ]:
grid = pd.DataFrame([dict(mouse_recday=mr, mouse=mr[:4],
                         **tp.grid_generalization(tp.build_taskphase_curves(data_dic, mr, config), config))
                     for mr in mouse_recdays
                     if tp.build_taskphase_curves(data_dic, mr, config)])
grid.to_pickle(f"{SAVE_DIR}/grid_{REGION}.pkl")
from scipy import stats
gsub = grid.dropna(subset=['grid_coherence','grid_coherence_null'])
print(f"coherence {gsub['grid_coherence'].mean():.3f} vs null {gsub['grid_coherence_null'].mean():.3f}, "
      f"shift {grid['grid_shift'].mean():.2f} rad, p={stats.wilcoxon(gsub['grid_coherence'], gsub['grid_coherence_null']).pvalue:.2g}")
tp.plot_grid_generalization(grid, region=REGION, out_path=f"{SAVE_DIR}/grid_{REGION}.pdf"); plt.show()

## 7. How to read this

| pattern | reading |
|---|---|
| h4 ≫ base, survives trim, present raw-time, C3→cycle | genuine 4-fold sub-goal cycle |
| h4 ≫ base but dies under trim, phase≈reward | reward-boundary transient |
| decay≈0.25, C3→arc | goal-progress ramp (open) |
| h4 no more than `state` synthetic | warp floor only |
| progress > null while state-CCGP ≈ null | progress abstract, identity not |